# CLUSE-Test on Kaggle — EvoEval 500

This notebook follows the project Kaggle Quick Start:

- sync project code from GitHub
- keep the 500-task EvoEval Parquet as a separate Kaggle Dataset input
- enable Kaggle Internet for GitHub and OpenAI/Hugging Face access
- load secrets from Kaggle Secrets
- run the mock smoke test before paid evaluation
- run five resumable 100-task shards
- merge shard results with `scripts/merge_shards.py`
- inspect the merged `report/` before treating results as final

Mock results are integration checks only and must not be reported as research results.


## 1. Sync the project from GitHub

Set `GITHUB_REPO` below to your repository. Kaggle Internet must be enabled.

This cell does not delete `/kaggle/working`, so completed shard outputs remain available when a session is resumed.


In [ ]:
import os
import subprocess
from pathlib import Path

PROJECT_DIR = Path("/kaggle/working/CLUSE-Test-EvoEval")

GITHUB_REPO = os.environ.get(
    "CLUSE_GITHUB_REPO",
    "https://github.com/antineutrin0/clus_test.git",
)

if PROJECT_DIR.exists() and (PROJECT_DIR / "run_pipeline.py").exists():
    print("Existing project found. Pulling latest changes...")

    subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"],
        check=True,
    )
else:
    print("Cloning project...")

    subprocess.run(
        ["git", "clone", GITHUB_REPO, str(PROJECT_DIR)],
        check=True,
    )

os.chdir(PROJECT_DIR)

print("Project:", PROJECT_DIR)
print("Git commit:")

subprocess.run(
    ["git", "rev-parse", "--short", "HEAD"],
    check=True,
)

Cloning project...


Cloning into '/kaggle/working/CLUSE-Test-EvoEval'...


Project: /kaggle/working/CLUSE-Test-EvoEval
Git commit:
eb6742b


CompletedProcess(args=['git', 'rev-parse', '--short', 'HEAD'], returncode=0)

## 2. Install dependencies

Keep Kaggle Internet **On**.


In [2]:
%pip install -q -r requirements.txt


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 43.1 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.


## 3. Load Kaggle secrets safely

Create and enable:

- `OPENAI_API_KEY`
- `HF_TOKEN` (optional but useful for Hugging Face)


In [3]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
except Exception as exc:
    secrets = None
    print("Kaggle Secrets client unavailable:", type(exc).__name__)

for secret_name in ("OPENAI_API_KEY", "HF_TOKEN"):
    if secrets is None:
        continue

    try:
        value = secrets.get_secret(secret_name)
    except Exception:
        value = None

    if value:
        os.environ[secret_name] = value
        print(f"{secret_name}: loaded")
    else:
        print(f"{secret_name}: not configured")

print("OpenAI key available:", bool(os.environ.get("OPENAI_API_KEY")))
print("HF token available:", bool(os.environ.get("HF_TOKEN")))


OPENAI_API_KEY: loaded
HF_TOKEN: loaded
OpenAI key available: True
HF token available: True


## 4. Attach and locate the finalized EvoEval dataset

The 500-task Parquet is a separate Kaggle Dataset input. This notebook does not rebuild the dataset.


In [4]:
from pathlib import Path

DATASET_DIR = Path("/kaggle/input/datasets/maharajhossain40/evoeval-dataset")
DATASET_PATH = DATASET_DIR / "EvoEval_semantic_500 (1).parquet"
DATASET_MANIFEST = DATASET_DIR / "EvoEval_semantic_500_manifest.json"

if not DATASET_PATH.exists():
    candidates = sorted(DATASET_DIR.glob("*.parquet"))
    if len(candidates) == 1:
        DATASET_PATH = candidates[0]
    else:
        raise FileNotFoundError(
            f"Could not uniquely locate a Parquet file under {DATASET_DIR}"
        )

print("Dataset:", DATASET_PATH)
print("Exists:", DATASET_PATH.exists())
print("Size (MB):", round(DATASET_PATH.stat().st_size / (1024**2), 2))
print("Manifest exists:", DATASET_MANIFEST.exists())


Dataset: /kaggle/input/datasets/maharajhossain40/evoeval-dataset/EvoEval_semantic_500 (1).parquet
Exists: True
Size (MB): 128.23
Manifest exists: False


## 5. Validate the finalized dataset


In [5]:
import pandas as pd
from IPython.display import display

metadata_df = pd.read_parquet(
    DATASET_PATH,
    columns=[
        "task_id",
        "source_task_id",
        "parent_task_id",
        "evoeval_subset",
        "entry_point",
    ],
)

print("Rows:", len(metadata_df))
print("Unique task IDs:", metadata_df["task_id"].nunique())
print("Unique HumanEval parents:", metadata_df["parent_task_id"].nunique())
print()
print("Subset distribution:")
display(
    metadata_df.groupby("evoeval_subset")
    .size()
    .rename("tasks")
    .reset_index()
)

assert len(metadata_df) == 500
assert metadata_df["task_id"].nunique() == 500
assert set(metadata_df.groupby("evoeval_subset").size()) == {100}
assert metadata_df["parent_task_id"].notna().all()

print("Dataset validation passed.")


Rows: 500
Unique task IDs: 500
Unique HumanEval parents: 100

Subset distribution:


,evoeval_subset,tasks
0,combine,100
1,creative,100
2,difficult,100
3,subtle,100
4,tool_use,100


Dataset validation passed.


## 6. Inspect normalized CLUSE-Test tasks


In [6]:
from src.utils.dataset_loader import load_dataset

DATASET_TYPE = "evoeval"
problems = load_dataset(DATASET_PATH, dataset_type=DATASET_TYPE)
first = problems[0]

print("Normalized tasks:", len(problems))
print("Dataset:", first.dataset_name)
print("Subset:", first.dataset_subset)
print("Task ID:", first.task_id)
print("Source task ID:", first.source_task_id)
print("Parent task ID:", first.parent_task_id)
print("Entry point:", first.entry_point)
print()
print("Specification preview:")
print(first.prompt_text[:1200])

assert len(problems) == 500


[2026-08-28 12:50:54] [INFO] [src.utils.dataset_loader] Loaded 500 normalized problems from /kaggle/input/datasets/maharajhossain40/evoeval-dataset/EvoEval_semantic_500 (1).parquet (dataset=evoeval)
Normalized tasks: 500
Dataset: evoeval
Subset: difficult
Task ID: EvoEval_difficult/0
Source task ID: EvoEval/0
Parent task ID: HumanEval/0
Entry point: has_close_elements_in_range

Specification preview:
Check if in given list of pairs of numbers, are any two consecutive pairs where the difference between 
the first elements and the difference between the second elements of the pairs are both less than 
the given threshold. Also, the pairs need to be sorted by the first element in ascending order before performing the check.

>>> has_close_elements_in_range([(1.0, 2.0), (2.0, 3.0), (3.0, 4.0)], 0.5)
False
>>> has_close_elements_in_range([(1.0, 2.8), (2.9, 3.0), (4.0, 5.0), (2.0, 2.1)], 0.3)
False


## 7. Configure the LLMs

Recommended shard/full-evaluation configuration:

- Layer 1: local Qwen
- Layer 2: OpenAI `gpt-5-mini`
- Layer 3: OpenAI `gpt-5-mini`
- Baseline: OpenAI `gpt-5-mini`

A token setting of `0` means no project-imposed output ceiling.


In [7]:
LLM_CONFIG = {
    "layer1_provider": "hf",
    "layer1_model": "Qwen/Qwen2.5-Coder-3B-Instruct",

    "layer2_provider": "openai",
    "layer2_model": "gpt-5-mini",

    "layer3_provider": "openai",
    "layer3_model": "gpt-5.2",

    "baseline_provider": "openai",
    "baseline_model": "gpt-5.2",

    "openai_reasoning_effort": "medium",
    "openai_text_verbosity": "low",

    "output_token_limit": 0,
}

LLM_CONFIG


{'layer1_provider': 'hf',
 'layer1_model': 'Qwen/Qwen2.5-Coder-3B-Instruct',
 'layer2_provider': 'openai',
 'layer2_model': 'gpt-5-mini',
 'layer3_provider': 'openai',
 'layer3_model': 'gpt-5.2',
 'baseline_provider': 'openai',
 'baseline_model': 'gpt-5.2',
 'openai_reasoning_effort': 'medium',
 'openai_text_verbosity': 'low',
 'output_token_limit': 0}

## 8. Verify configured OpenAI model IDs


In [8]:
if os.environ.get("OPENAI_API_KEY"):
    from openai import OpenAI

    client = OpenAI()
    available_model_ids = {model.id for model in client.models.list().data}

    for key in ("layer2_model", "layer3_model", "baseline_model"):
        model_id = LLM_CONFIG[key]
        print(
            f"{key}: {model_id} ->",
            "AVAILABLE" if model_id in available_model_ids else "NOT LISTED",
        )
else:
    print("OPENAI_API_KEY is not loaded; model availability check skipped.")


layer2_model: gpt-5-mini -> AVAILABLE
layer3_model: gpt-5.2 -> AVAILABLE
baseline_model: gpt-5.2 -> AVAILABLE


## 9. Run the no-cost integration smoke test

Run this before any paid shard.

The mock run checks EvoEval normalization, mutation generation, clustering, batched execution, statistics, figures, and ZIP creation without API charges.


In [10]:
import subprocess
import sys
from pathlib import Path

SMOKE_RESULTS = Path("/kaggle/working/cluse_evoeval_smoke")

smoke_command = [
    sys.executable,
    "run_pipeline.py",
    "--dataset", str(DATASET_PATH),
    "--dataset-type", DATASET_TYPE,
    "--run-name", "evoeval_smoke",
    "--limit", "1",
    "--sample-mode", "stratified",
    "--stratify-by", "dataset_subset",
    "--seed", "42",
    "--results", str(SMOKE_RESULTS),
    "--max-layers", "3",
    "--max-mutants", "5",
    "--max-probes", "3",
    "--mock",
    "--layer1-max-tokens", "0",
    "--layer2-max-tokens", "0",
    "--layer3-max-tokens", "0",
    "--openai-reasoning-effort", "high",
    "--openai-text-verbosity", "low",
    "--display-llm-responses", "1",
    "--display-first-problem-only", "1",
    "--display-compact-call-summary", "1",
    "--save-llm-responses", "1",
    "--generate-statistics", "1",
    "--bootstrap-samples", "500",
    "--figure-pdf", "0",
    "--zip-output",
]

print(" ".join(smoke_command))
subprocess.run(smoke_command, check=True)
print("Smoke test completed.")


/usr/bin/python3 run_pipeline.py --dataset /kaggle/input/datasets/maharajhossain40/evoeval-dataset/EvoEval_semantic_500 (1).parquet --dataset-type evoeval --run-name evoeval_smoke --limit 1 --sample-mode stratified --stratify-by dataset_subset --seed 42 --results /kaggle/working/cluse_evoeval_smoke --max-layers 3 --max-mutants 5 --max-probes 3 --mock --layer1-max-tokens 0 --layer2-max-tokens 0 --layer3-max-tokens 0 --openai-reasoning-effort high --openai-text-verbosity low --display-llm-responses 1 --display-first-problem-only 1 --display-compact-call-summary 1 --save-llm-responses 1 --generate-statistics 1 --bootstrap-samples 500 --figure-pdf 0 --zip-output
[2026-08-28 07:28:51] [INFO] [src.utils.dataset_loader] Loaded 500 normalized problems from /kaggle/input/datasets/maharajhossain40/evoeval-dataset/EvoEval_semantic_500 (1).parquet (dataset=evoeval)
[2026-08-28 07:28:51] [INFO] [src.pipeline] Run evoeval_smoke: selected 1/500 problems, max_layers=3, max_mutants=5, max_probes=3
[202

## 10. Select the 100-task shard

The full 500-task evaluation is run as five resumable shards:

- shard 0: 0–99
- shard 1: 100–199
- shard 2: 200–299
- shard 3: 300–399
- shard 4: 400–499

Change `SHARD_INDEX` before running the shard cell.


In [9]:
RUN_SHARD = True
SHARD_INDEX = 0

SHARDS = [
    (0, 50, "/kaggle/working/results_shard0"),
    (50, 200, "/kaggle/working/results_shard1"),
    (200, 300, "/kaggle/working/results_shard2"),
    (300, 400, "/kaggle/working/results_shard3"),
    (400, 500, "/kaggle/working/results_shard4"),
]

index_start, index_end, RESULTS_SHARD = SHARDS[SHARD_INDEX]

print("Shard:", SHARD_INDEX)
print("Index range:", index_start, "to", index_end - 1)
print("Results:", RESULTS_SHARD)


Shard: 0
Index range: 0 to 49
Results: /kaggle/working/results_shard0


## 11. Run the selected 100-task shard

Save the Kaggle notebook version after each completed shard.

The shard uses the index-range interface described in the Kaggle Quick Start.


In [ ]:
import subprocess
import sys
from pathlib import Path

real_command = [
    sys.executable,
    "run_pipeline.py",

    "--evoeval-semantic",
    "--hf-split", "test",
    "--dataset-type", "evoeval",

    "--index-start", str(index_start),
    "--index-end", str(index_end),

    "--results", str(RESULTS_SHARD),

    # Layer 1
    "--layer1-provider", LLM_CONFIG["layer1_provider"],
    "--layer1-model", LLM_CONFIG["layer1_model"],
    "--layer1-max-tokens", "0",
    "--layer1-max-attempts", "3",
    "--layer1-plateau-patience", "2",
    "--layer1-stop-on-plateau", "0",

    # Layer 2
    "--layer2-provider", LLM_CONFIG["layer2_provider"],
    "--layer2-model", LLM_CONFIG["layer2_model"],
    "--layer2-max-attempts", "10",
    "--layer2-max-tokens", "0",
    "--layer2-plateau-patience", "1",
    "--layer2-stop-on-plateau", "1",

    # Layer 3
    "--layer3-provider", LLM_CONFIG["layer3_provider"],
    "--layer3-model", LLM_CONFIG["layer3_model"],
    "--layer3-max-attempts", "10",
    "--layer3-max-tokens", "0",
    "--layer3-plateau-patience", "1",
    "--layer3-stop-on-plateau", "1",

    # Baseline
    "--run-baseline",
    "--baseline-provider", LLM_CONFIG["baseline_provider"],
    "--baseline-model", LLM_CONFIG["baseline_model"],
    "--baseline-max-iterations", "10",
    "--baseline-max-tokens", "0",
    "--baseline-plateau-patience", "2",
    "--baseline-stop-on-plateau", "1",

    "--require-productive-test", "1",

    # OpenAI
    "--openai-reasoning-effort", LLM_CONFIG["openai_reasoning_effort"],
    "--openai-text-verbosity", LLM_CONFIG["openai_text_verbosity"],

    # Logging
    "--display-llm-responses", "1",
    "--display-first-problem-only", "1",
    "--display-compact-call-summary", "1",
    "--log-full-llm-io", "0",
    "--save-llm-responses", "0",

    # Statistics
    "--generate-statistics", "1",
    "--bootstrap-samples", "5000",
    "--figure-pdf", "1",
    "--zip-output",
]

print(" ".join(real_command))

if RUN_SHARD:
    subprocess.run(real_command, check=True)
else:
    print("Shard not executed. Set RUN_SHARD=True after smoke/model checks pass.")


/usr/bin/python3 run_pipeline.py --evoeval-semantic --hf-split test --dataset-type evoeval --index-start 0 --index-end 50 --results /kaggle/working/results_shard0 --layer1-provider hf --layer1-model Qwen/Qwen2.5-Coder-3B-Instruct --layer1-max-tokens 0 --layer1-max-attempts 3 --layer1-plateau-patience 2 --layer1-stop-on-plateau 0 --layer2-provider openai --layer2-model gpt-5-mini --layer2-max-attempts 10 --layer2-max-tokens 0 --layer2-plateau-patience 1 --layer2-stop-on-plateau 1 --layer3-provider openai --layer3-model gpt-5.2 --layer3-max-attempts 10 --layer3-max-tokens 0 --layer3-plateau-patience 1 --layer3-stop-on-plateau 1 --run-baseline --baseline-provider openai --baseline-model gpt-5.2 --baseline-max-iterations 10 --baseline-max-tokens 0 --baseline-plateau-patience 2 --baseline-stop-on-plateau 1 --require-productive-test 1 --openai-reasoning-effort medium --openai-text-verbosity low --display-llm-responses 1 --display-first-problem-only 1 --display-compact-call-summary 1 --log-fu

Loading weights: 100%|██████████| 434/434 [00:02<00:00, 206.41it/s, Materializing param=model.norm.weight]                              
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



==================== LLM RESPONSE 1 ====================
problem=EvoEval_difficult/0 layer=Layer1 attempt=1 provider=hf model=Qwen/Qwen2.5-Coder-3B-Instruct targets=5
tokens=3557 new_kills=1 cumulative_score=0.048 status=PRODUCTIVE
```python
def check_candidate(candidate):
    assert candidate([1.0, 10.0, 20.0, 30.0], 5.0) == False  # New assertion based on the provided examples
    assert candidate([10.0, -10.0,20.0,30.0], -5.0) == True  # New assertion based observation
    assert candidate([-10.0,10.0,-20.0,15.0], -5) == True  # Additional assertion based on observation
    assert candidate([0.0, -0.0, 0.0, 50.0], 0) == True  #
```


==================== LLM RESPONSE 2 ====================
problem=EvoEval_difficult/0 layer=Layer1 attempt=2 provider=hf model=Qwen/Qwen2.5-Coder-3B-Instruct targets=5
tokens=3890 new_kills=0 cumulative_score=0.048 status=VALID_ZERO_KILL
```python
def check_candidate(candidate):
    assert candidate([1.0, 10.0, 20.0, 30.0], 5.0) == False  # New assertio

## 12. Merge the five shard results

Run after all five shards complete.


In [18]:
from pathlib import Path
import subprocess
import sys

SHARD_DIRS = [
    Path("/kaggle/working/results_shard0"),
    # Path("/kaggle/working/results_shard1"),
    # Path("/kaggle/working/results_shard2"),
    # Path("/kaggle/working/results_shard3"),
    # Path("/kaggle/working/results_shard4"),
]

RESULTS_FULL = Path("/kaggle/working/results_full")

missing = [str(path) for path in SHARD_DIRS if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing shard directories:\n" + "\n".join(missing)
    )

merge_command = [
    sys.executable,
    str(PROJECT_DIR / "scripts" / "merge_shards.py"),
    "--shards",
    *[str(path) for path in SHARD_DIRS],
    "--output",
    str(RESULTS_FULL),
]

print(" ".join(merge_command))
subprocess.run(merge_command, check=True)
print("Merged results:", RESULTS_FULL)


/usr/bin/python3 /kaggle/working/CLUSE-Test-EvoEval/scripts/merge_shards.py --shards /kaggle/working/results_shard0 --output /kaggle/working/results_full
Merging raw artifacts from /kaggle/working/results_shard0/raw -> /kaggle/working/results_full/raw
[2026-08-28 09:37:49] [INFO] [src.evaluation.statistical_analysis] Saved statistical report to /kaggle/working/results_full/report/statistics and figures to /kaggle/working/results_full/report/figures
[2026-08-28 09:37:49] [INFO] [src.pipeline] === Run merged: no executed problems (skipped=0) ===
{
  "run_name": "merged",
  "shards": [
    "/kaggle/working/results_shard0"
  ],
  "shard_manifests": [
    {
      "shard_dir": "/kaggle/working/results_shard0"
    }
  ],
  "selected_problems": 0,
  "executed_problems": 0,
  "skipped_problems": 0,
  "dataset_names": [],
  "dataset_subsets": [],
  "subset_counts": {},
  "llm_configuration": {}
}
Merged results: /kaggle/working/results_full


## 14. Inspect statistical tables


In [ ]:
import pandas as pd
from IPython.display import display

STATS_DIR = RESULTS_FULL / "report" / "statistics"

for filename in [
    "effectiveness_summary.csv",
    "paired_comparison_summary.csv",
    "evoeval_subset_summary.csv",
    "evoeval_subset_paired_comparison.csv",
    "efficiency_summary.csv",
    "layer_contribution_summary.csv",
    "operator_difficulty_summary.csv",
    "llm_usage_summary.csv",
]:
    path = STATS_DIR / filename

    if path.exists():
        print()
        print(filename)
        display(pd.read_csv(path))
    else:
        print("Missing:", filename)


## 15. Inspect saved figures


In [ ]:
from IPython.display import Image, display

FIGURES_DIR = RESULTS_FULL / "report" / "figures"
figures = sorted(FIGURES_DIR.glob("*.png"))

print("Saved figures:", len(figures))

for figure in figures:
    print(figure.name)
    display(Image(filename=str(figure)))


## 16. Debugging a specific shard

For a run where you need to inspect exactly what a model saw, add:

`--verbose-artifacts 1`

to the shard command. This writes detailed prompt/response artifacts under `raw/llm_responses/`.

Keep it off for normal pilot/full runs unless the archive is specifically required.


In [ ]:
DEBUG_VERBOSE = False

if DEBUG_VERBOSE:
    debug_command = real_command + ["--verbose-artifacts", "1"]
    print(" ".join(debug_command))
else:
    print("Verbose artifact logging disabled.")


## 17. Package the merged results


In [ ]:
import shutil

if not RESULTS_FULL.exists():
    raise FileNotFoundError(RESULTS_FULL)

zip_path = shutil.make_archive(
    str(RESULTS_FULL),
    "zip",
    root_dir=RESULTS_FULL,
)

print("ZIP:", zip_path)
print("Size (MB):", round(Path(zip_path).stat().st_size / (1024**2), 2))


In [ ]:
import shutil

if not RESULTS_FULL.exists():
    raise FileNotFoundError(RESULTS_FULL)

zip_path = shutil.make_archive(
    str(RESULTS_FULL),
    "zip",
    root_dir=RESULTS_FULL,
)

print("ZIP:", zip_path)
print("Size (MB):", round(Path(zip_path).stat().st_size / (1024**2), 2))

## 18. Research interpretation checklist

Before reporting the final experiment:

- report macro and micro mutation scores together
- use paired confidence intervals, exact sign test, effect sizes, and win/tie/loss counts
- use parent-clustered confidence intervals when HumanEval parents are shared
- report per-subset effectiveness and token efficiency
- report marginal kills by layer
- report paid tokens and cost per killed mutant
- report runtime breakdown
- compare fixed-budget and plateau-stopped baseline settings
- keep official-test agreement separate from mutation effectiveness
- confirm `reclustering_disabled=true`
- report how often Layer 2/Layer 3 were skipped because an earlier layer killed all mutants
